In [1]:
pip install yfinance pandas numpy matplotlib

In [3]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from typing import List, Tuple
import gradio as gr


# -------------------------------
# 0) Fetching (parametric interval/period)
# -------------------------------

def fetch_ohlc(
    ticker: str,
    interval: str = "1h",
    period: str = "1y",
    tz: str = "America/Toronto",
    prepost: bool = True,
    auto_adjust: bool = False,
) -> pd.DataFrame:
    """
    Fetch OHLC for `ticker` with parametric interval/period.
    Guarantees returned df has flat columns: ['High', 'Low', 'Close'].
    Handles:
      - MultiIndex columns like ('High','AAPL')
      - Suffixed columns like 'High_AAPL'
      - Plain columns 'High','Low','Close'
    """
    df = yf.download(
        ticker,
        #start="2025-05-01",
        #end="2026-04-27",
        interval=interval,
        period=period,
        prepost=prepost,
        auto_adjust=auto_adjust,
        progress=False,
        # NOTE: Some yfinance versions ignore group_by here; we handle both shapes below.
        # group_by="column",
    )

    if df is None or df.empty:
        raise ValueError(f"No data for {ticker} interval={interval} period={period}")

    # Ensure tz-aware index before converting
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    df.index = df.index.tz_convert(tz)

    # --- Normalize columns ---
    if isinstance(df.columns, pd.MultiIndex):
        # Typical shape: level 0 = field, level 1 = ticker
        # Select the desired ticker at level=1
        tickers_in_df = df.columns.get_level_values(1).unique()
        if len(tickers_in_df) == 1:
            # Only one ticker; drop level 1 to leave plain field names
            df = df.droplevel(1, axis=1)
        else:
            # Multiple tickers present; select the one we asked for
            df = df.xs(ticker, axis=1, level=1)
        # After this, df.columns should be like ['Open','High','Low','Close','Adj Close','Volume']

    # If still not plain field names, handle suffixed columns like 'High_AAPL'
    cols = list(df.columns)
    required = {"High", "Low", "Close"}
    if not required.issubset(set(cols)):
        # Try to map suffixed names to base names
        def find_col(base: str) -> str | None:
            # Exact name
            for c in cols:
                if str(c).strip().lower() == base.lower():
                    return c
            # Suffix: Base_TICKER (e.g., High_AAPL)
            candidates = [c for c in cols if str(c).strip().lower().startswith(base.lower())]
            if candidates:
                # Prefer exact suffix match with ticker
                for c in candidates:
                    parts = str(c).split("_")
                    if len(parts) >= 2 and parts[-1].upper() == ticker.upper():
                        return c
                # Else, take the first candidate
                return candidates[0]
            return None

        c_high = find_col("High")
        c_low  = find_col("Low")
        c_close = find_col("Close")
        if not all([c_high, c_low, c_close]):
            raise KeyError(f"Could not find High/Low/Close in columns: {cols}")

        df = df[[c_high, c_low, c_close]].copy()
        df.columns = ["High", "Low", "Close"]

    else:
        # Keep only needed columns
        df = df[["High", "Low", "Close"]].copy()

    # Final cleanup
    df = df.dropna()
    df = df[~df.index.duplicated(keep="first")].sort_index()

    #print(df.tail())
    return df


# -------------------------------
# 1) Aggregate intraday bars to session (daily)
# -------------------------------

def aggregate_to_session(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate intraday bars into session-level OHLC features (local calendar day):
      SessionHigh  = max(High)
      SessionLow   = min(Low)
      SessionClose = last(Close) of the day (final intraday bar)
    Returns DataFrame indexed by local calendar date with columns:
      ['SessionHigh', 'SessionLow', 'SessionClose']
    """
    temp = df.copy()
    temp["SessionKey"] = pd.to_datetime(temp.index.date)

    agg = temp.groupby("SessionKey").agg(
        SessionHigh=("High", "max"),
        SessionLow=("Low", "min"),
        SessionClose=("Close", "last"),
    )
    agg = agg.sort_index()
    return agg


def attach_prev_session_close_to_sessions(session_df: pd.DataFrame) -> pd.DataFrame:
    """
    Attach previous session close as 'PrevSessionClose' to the session rows.
    """
    out = session_df.copy()
    out["PrevSessionClose"] = out["SessionClose"].shift(1)
    out = out.dropna(subset=["PrevSessionClose"])
    return out


# -------------------------------
# 2) State labeling (one state per session/day)
# -------------------------------

def label_states_session_level(
    session_with_prev: pd.DataFrame,
    up_thresh: float = 0.02,   # 2% or more higher => S1 condition
    down_thresh: float = 0.01  # 1% or more lower => S2 condition
) -> pd.DataFrame:
    """
    Label each session/day with 7 mutually-exclusive states using:
      S1: SessionHigh >= PrevSessionClose * (1 + up_thresh)
      S2: SessionLow  <= PrevSessionClose * (1 - down_thresh)
      S3: SessionClose in [Prev*(1 - down_thresh), Prev*(1 + up_thresh))  (inside the band)
      S4: S1 and S2
      S5: S1 and S3
      S6: S2 and S3
      S7: S1 and S2 and S3
    """
    df = session_with_prev.copy()

    prev = df["PrevSessionClose"].to_numpy()
    hi   = df["SessionHigh"].to_numpy()
    lo   = df["SessionLow"].to_numpy()
    clo  = df["SessionClose"].to_numpy()

    cond1 = hi >= prev * (1 + up_thresh)
    cond2 = lo <= prev * (1 - down_thresh)
    cond3 = (clo > prev * (1 - down_thresh)) & (clo < prev * (1 + up_thresh))

    # Bitmask via arithmetic (Series-friendly)
    bitmask = (cond1.astype(int) * 1) + (cond2.astype(int) * 2) + (cond3.astype(int) * 4)

    mapping = {1: 1, 2: 2, 4: 3, 3: 4, 5: 5, 6: 6, 7: 7}
    state = np.full(bitmask.shape, np.nan)
    for k, v in mapping.items():
        state[bitmask == k] = v

    df["State"] = pd.Series(state, index=df.index, dtype="Int64")
    df = df.dropna(subset=["State"])
    df["State"] = df["State"].astype(int)
    return df

# -------------------------------
# 2.5) Add log-excursion columns (relative to prev close)
# -------------------------------

def add_excursion_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds log excursions relative to yesterday's close.
    """
    out = df.copy()

    out["log_high"]  = np.log(out["SessionHigh"]  / out["PrevSessionClose"])
    out["log_low"]   = np.log(out["SessionLow"]   / out["PrevSessionClose"])
    out["log_close"] = np.log(out["SessionClose"] / out["PrevSessionClose"])

    return out


# -------------------------------
# 3) Markov chain fitting (session -> session)
# -------------------------------

def fit_markov(
    labeled_dfs: List[pd.DataFrame],
    num_states: int = 7,
    laplace_alpha: float = 1.0
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Fit a Markov chain from one or more labeled session DataFrames (with 'State' column).
    Transitions are counted within each DataFrame, then aggregated.
    Laplace smoothing avoids zero-prob transitions.
    """
    counts = np.zeros((num_states, num_states), dtype=np.float64)

    for df in labeled_dfs:
        states = df["State"].astype(int).to_numpy()
        for i in range(len(states) - 1):
            s_t, s_tp1 = states[i], states[i + 1]
            if 1 <= s_t <= num_states and 1 <= s_tp1 <= num_states:
                counts[s_t - 1, s_tp1 - 1] += 1

    smoothed = counts + laplace_alpha
    row_sums = smoothed.sum(axis=1, keepdims=True)
    probs = smoothed / np.where(row_sums == 0, 1.0, row_sums)
    return counts, probs

# -------------------------------
# Fit higher-order Markov (history-aware)
# -------------------------------
from collections import defaultdict

def fit_higher_order_markov(labeled_df, order=3, num_states=7, alpha=1.0):
    counts = defaultdict(lambda: np.full(num_states, alpha))
    states = labeled_df["State"].astype(int).to_list()

    for i in range(len(states) - order):
        history = tuple(states[i:i+order])
        next_state = states[i + order]
        counts[history][next_state - 1] += 1

    probs = {h: v / v.sum() for h, v in counts.items()}
    return probs

def predict_next_state(model, history, fallback_probs):
    history = tuple(history)
    if history in model:
        dist = model[history]
    else:
        dist = fallback_probs

    next_state = int(np.argmax(dist) + 1)
    next_prob = float(np.max(dist))
    return next_state, next_prob,



def compute_belief_vector(states, num_states=7, decay=0.97):
    """
    Build a normalized belief vector over states using exponentially
    decaying weights over ALL past history.
    """
    belief = np.zeros(num_states, dtype=np.float64)

    weight = 1.0
    for s in reversed(states):
        belief[s - 1] += weight    # - 1 for indexing
        weight *= decay

    belief /= belief.sum()
    return belief

def predict_from_belief(belief, transition_matrix):
    """
    Predict next-state distribution from belief vector.
    """
    next_dist = belief @ transition_matrix
    next_state = int(np.argmax(next_dist) + 1)
    next_prob = float(np.max(next_dist))
    return next_state, next_prob, next_dist

# -------------------------------
# 4) Helpers (printing & heatmap)
# -------------------------------

def print_matrix(name: str, mat: np.ndarray, precision: int = 4):
    headers = [f"S{j}" for j in range(1, mat.shape[1] + 1)]
    print(f"\n{name}:")
    print("      " + "  ".join([f"{h:>6}" for h in headers]))
    for i in range(mat.shape[0]):
        row = "  ".join([f"{v:>6.{precision}f}" for v in mat[i]])
        print(f"S{i+1}: {row}")


def plot_heatmap(probs: np.ndarray, title: str = "Transition Probabilities (Session → Session)"):
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(probs, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("Next state")
    ax.set_ylabel("Current state")
    ax.set_xticks(range(probs.shape[1])); ax.set_xticklabels([f"S{i}" for i in range(1, probs.shape[1] + 1)])
    ax.set_yticks(range(probs.shape[0])); ax.set_yticklabels([f"S{i}" for i in range(1, probs.shape[0] + 1)])
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    return fig

# -------------------------------
# State → implied excursions
# -------------------------------

STATE_IMPLIED_TARGETS = {
    1: ["log_high"],                        # S1: up
    2: ["log_low"],                         # S2: down
    3: ["log_close"],                       # S3: close
    4: ["log_high", "log_low"],             # S4: up + down
    5: ["log_high", "log_close"],           # S5: up + close
    6: ["log_low", "log_close"],            # S6: down + close
    7: ["log_high", "log_low", "log_close"] # S7: all
}

# -------------------------------
# Compute state-implied excursions
# -------------------------------

def compute_state_implied_excursions(df: pd.DataFrame) -> dict:
    """
    Computes implied log excursions for each state,
    storing raw values for quantile-based reporting.
    """
    result = {}

    for state, targets in STATE_IMPLIED_TARGETS.items():
        subset = df[df["State"] == state]

        if subset.empty:
            continue

        state_result = {}

        for col in targets:
            values = subset[col].values  # <-- KEEP RAW VALUES

            state_result[col] = {
                "values": values,
                "mean": values.mean(),
                "std": values.std(),
                "count": len(values)
            }

        result[state] = state_result

    return result

# -------------------------------
# Log-return interpretation helpers (REPORTING ONLY)
# -------------------------------

def log_to_percent(log_value: float) -> float:
    """
    Convert log-return to percent change.
    """
    return (np.exp(log_value) - 1) * 100


def log_mean_std_to_percent_range(mu: float, sigma: float):
    """
    Convert log-mean and log-std into an interpretable 1-sigma percent range.
    """
    lower = (np.exp(mu - sigma) - 1) * 100
    upper = (np.exp(mu + sigma) - 1) * 100
    return lower, upper


def log_quantile_to_percent_range(values, q_low=25, q_high=75):
    """
    Convert log-return quantiles into percent range.
    """
    lo = np.percentile(values, q_low)
    hi = np.percentile(values, q_high)
    return (np.exp(lo) - 1) * 100, (np.exp(hi) - 1) * 100

def run_model(
    tickers_text: str,
    up_thresh: float,
    down_thresh: float,
    decay: float
):
    try:
        # -------------------------------
        # Parse tickers
        # -------------------------------
        tickers = [t.strip().lower() for t in tickers_text.split(",") if t.strip()]
        if not tickers:
            return "No valid tickers entered.", None

        interval = "1d"
        period = "1y"
        tz = "America/Toronto"
        prepost = True
        auto_adjust = False
        laplace_alpha = 1.0

        labeled_list = []

        # -------------------------------
        # Build labeled session data
        # -------------------------------
        for t in tickers:
            bars = fetch_ohlc(
                t,
                interval=interval,
                period=period,
                tz=tz,
                prepost=prepost,
                auto_adjust=auto_adjust
            )

            sessions = aggregate_to_session(bars)
            sessions_prev = attach_prev_session_close_to_sessions(sessions)

            labeled_sessions = label_states_session_level(
                sessions_prev,
                up_thresh=up_thresh,
                down_thresh=down_thresh
            )

            labeled_sessions = add_excursion_columns(labeled_sessions)
            labeled_sessions["Ticker"] = t
            labeled_list.append(labeled_sessions)

        # -------------------------------
        # Fit Markov transition matrix
        # -------------------------------
        _, probs = fit_markov(
            labeled_list,
            num_states=7,
            laplace_alpha=laplace_alpha
        )

        # -------------------------------
        # Belief inference (full history)
        # -------------------------------
        states = labeled_sessions["State"].astype(int).to_list()

        belief = compute_belief_vector(
            states=states,
            num_states=7,
            decay=decay
        )

        _, _, next_dist = predict_from_belief(belief, probs)

        # -------------------------------
        # State‑implied excursions
        # -------------------------------
        state_excursions = compute_state_implied_excursions(labeled_sessions)


        # Get latest available date
        last_date = labeled_sessions.index.max()
        last_date_str = str(last_date.date())

        # -------------------------------
        # BUILD TEXT OUTPUT (NO PRINTS)
        # -------------------------------
        output = []
        output.append(f"Today (Latest availabe data date): {last_date_str}")
        output.append("")

        output.append("MODEL ASSUMPTION:")
        output.append(
            "S1: Up | S2: Down | S3: Inside band | "
            "S4: Up+Down | S5: Up+Close | S6: Down+Close | S7: All"
        )

        output.append("\nNEXT‑DAY STATE PROBABILITIES:")
        for i, p in enumerate(next_dist, start=1):
            output.append(f"  S{i}: {p:.4f}")

        output.append("\nSTATE‑IMPLIED EXCURSIONS (percent vs today’s close):")

        for state, targets in state_excursions.items():
            output.append(f"\nState S{state}:")

            for name, stats in targets.items():
                values = stats["values"]

                mean_pct = (np.exp(np.mean(values)) - 1) * 100
                lo, hi = log_quantile_to_percent_range(values, 25, 75)

                label = name.replace("log_", "").upper()

                output.append(
                    f"  {label}: expected {mean_pct:+.2f}% "
                    f"(typical {lo:+.2f}% → {hi:+.2f}%), n={len(values)}"
                )

        # -------------------------------
        # Heatmap (RETURN FIGURE)
        # -------------------------------
        fig = plot_heatmap(
            probs,
            title=f"Transition Probabilities — interval={interval}, period={period}"
        )

        return "\n".join(output), fig

    except Exception as e:
        return f"Error: {str(e)}", None


# -------------------------------
# Gradio UI
# -------------------------------

demo = gr.Interface(
    fn=run_model,
    inputs=[
        gr.Textbox(
            label="Tickers (comma-separated)",
            placeholder="e.g. NVDA, AAPL, MSFT",
            value="NVDA",
        ),
        gr.Number(
            label="Up Threshold",
            value=0.02,
            precision=4
        ),
        gr.Number(
            label="Down Threshold",
            value=0.01,
            precision=4
        ),
        gr.Slider(
            minimum=0.9,
            maximum=0.999,
            step=0.01,
            value=0.97,
            label="History Decay"
        )
    ],
    outputs=[
        gr.Textbox(
            label="Model Output",
            lines=25
        ),
        gr.Plot(
            label="Transition Heatmap"
        )
    ],
    title="Market State & Excursion Model",
    description=(
        "State-based market regime model.\n\n"
        "• Predicts next-day state probabilities\n"
        "• Computes state-implied intraday excursions\n"
        "• Uses exponentially weighted belief over history"
    )
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://37b4f6e1f968b2ff80.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
